# Prepare data for ML

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

business = pd.read_json("/content/drive/MyDrive/163/yelp_academic_dataset_business.json", lines=True)


In [ ]:
restaurants = business[
    business["categories"].str.contains("Restaurant", na=False)
]

In [ ]:
restaurants = restaurants[restaurants["review_count"] >= 50]


In [ ]:
restaurants["review_count"].min()

50

In [ ]:
# Import TextBlob for sentiment analysis.
# TextBlob assigns each review a polarity score:
#   -1.0 = very negative
#    0.0 = neutral
#   +1.0 = very positive
from textblob import TextBlob
import pandas as pd
# Import garbage collection to free memory after each chunk.
import gc

# Get the set of restaurant business IDs from our filtered restaurant dataset.
# We only want to process reviews that belong to restaurants in our analysis.
restaurant_ids = set(restaurants["business_id"])

# Create a list to store partial results from each chunk.
# Each chunk will produce:
#   business_id
#   sentiment_sum = sum of sentiment scores
#   review_n = number of reviews
chunk_results = []

# Read the Yelp review dataset in chunks of 10,000 rows at a time.
# This is necessary because the full review dataset is very large and
# may not fit into memory all at once.
for chunk in pd.read_json(
    "/content/drive/MyDrive/163/yelp_academic_dataset_review.json",
    lines=True,
    chunksize=10000
):
    # Keep only reviews for restaurants in our dataset.
    filtered = chunk[chunk["business_id"].isin(restaurant_ids)].copy()

    # If no reviews in this chunk match our restaurants, skip it.
    if filtered.empty:
        continue

    # Compute sentiment for each review.
    # TextBlob returns a polarity score between -1 and 1.
    filtered["sentiment"] = filtered["text"].apply(
        lambda x: TextBlob(str(x)).sentiment.polarity
    )

    # Aggregate sentiment within this chunk.
    # For each restaurant:
    #   sentiment_sum = total sentiment across all reviews in the chunk
    #   review_n = number of reviews in the chunk
    agg_chunk = filtered.groupby("business_id").agg(
        sentiment_sum=("sentiment", "sum"),
        review_n=("sentiment", "size")
    ).reset_index()

    # Save this chunk's aggregated results.
    chunk_results.append(agg_chunk)

    # Delete temporary objects to free memory.
    del chunk, filtered, agg_chunk
    gc.collect()

# combine all chunks into one DataFrame.
sentiment_df = pd.concat(chunk_results)

# Aggregate again across all chunks.
# Since the same restaurant may appear in multiple chunks,
# we sum the sentiment totals and review counts.
sentiment_df = sentiment_df.groupby("business_id").agg(
    sentiment_sum=("sentiment_sum", "sum"),
    review_n=("review_n", "sum")
).reset_index()

# Compute the final average sentiment for each restaurant.
sentiment_df["avg_sentiment"] = (
    sentiment_df["sentiment_sum"] / sentiment_df["review_n"]
)

# Merge the average sentiment feature into the main restaurant dataset.
# Each restaurant now has one additional column:
#   avg_sentiment
restaurants = restaurants.merge(
    sentiment_df[["business_id", "avg_sentiment"]],
    on="business_id",
    how="left"
)

## Complaint Frequency

In [ ]:
# List of words that indicate possible customer complaints.
# These are terms commonly found in negative reviews.
complaint_words = [
    "slow", "rude", "overpriced", "bad", "terrible",
    "awful", "cold", "bland", "dirty", "wait"
]

In [ ]:
# Function to count how many complaint words appear in a review.
def count_complaints(text):
    # Convert the review text to lowercase so matching is case-insensitive.
    text = str(text).lower()

    # For each complaint word, check whether it appears in the review.
    # True counts as 1 and False counts as 0.
    # Example:
    # "slow service and rude staff" -> contains "slow" and "rude" -> returns 2
    return sum(word in text for word in complaint_words)

In [ ]:
# Get all restaurant business IDs from our restaurant dataset.
# We only want to process reviews for these restaurants.
restaurant_ids = set(restaurants["business_id"])


# Dictionary to store the total number of complaint words
# across all reviews for each restaurant.
# Example:
# complaint_totals["abc123"] = 42
complaint_totals = {}


# Dictionary to store the total number of reviews
# processed for each restaurant.
# Example:
# review_counts["abc123"] = 100
review_counts = {}


# Read the Yelp review dataset in chunks of 10,000 rows.
# This avoids loading the entire large review file into memory at once.
for chunk in pd.read_json(
    "/content/drive/MyDrive/163/yelp_academic_dataset_review.json",
    lines=True,
    chunksize=10000
):

    # Keep only reviews belonging to restaurants in our dataset.
    chunk = chunk[chunk["business_id"].isin(restaurant_ids)]


    # Compute the complaint score for each review.
    # This is the number of complaint words found in the review text.
    chunk["complaints"] = chunk["text"].apply(count_complaints)


    # Process each review in the chunk.
    for _, row in chunk.iterrows():
        # Get the restaurant ID.
        bid = row["business_id"]

        # Add this review's complaint score to the restaurant's total.
        # If the restaurant is not already in the dictionary, start at 0.
        complaint_totals[bid] = (
            complaint_totals.get(bid, 0) + row["complaints"]
        )

        # Increase the review count for this restaurant by 1.
        review_counts[bid] = (
            review_counts.get(bid, 0) + 1
        )


In [ ]:
# Convert total complaints into average complaints per review.
# Formula:
#   avg_complaints = total complaint words / number of reviews
#
# Example:
#   30 complaint words across 100 reviews -> 0.30
avg_complaints = {
    bid: complaint_totals[bid] / review_counts[bid]
    for bid in complaint_totals
}


In [ ]:
# Convert the dictionary into a DataFrame.
# The business_id becomes the index initially.
complaints_df = pd.DataFrame.from_dict(
    avg_complaints,
    orient="index",
    columns=["avg_complaints"]
).reset_index()


# Rename the index column to business_id.
complaints_df = complaints_df.rename(
    columns={"index": "business_id"}
)


# Merge the average complaint feature into the main restaurant dataset.
# Each restaurant now gets one additional column:
#   avg_complaints
restaurants = restaurants.merge(
    complaints_df,
    on="business_id",
    how="left"
)

# If a restaurant did not receive a complaint score
# (for example, if no reviews were processed),
# replace missing values with 0.
restaurants["avg_complaints"] = (
    restaurants["avg_complaints"].fillna(0)
)

#Machine Learning

### Model 1: Review count only

In [ ]:
#ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

#choose feature
data = restaurants[["is_open", "log_reviews", "stars", "avg_sentiment", "avg_complaints"]].dropna()

y = data["is_open"]
X1 = data[["log_reviews"]]

#split into train/test sets
X1_train, X1_test, y_train, y_test = train_test_split(
    X1, y, test_size=0.2, random_state=42, stratify=y
)

#scale features
scaler1 = StandardScaler()
X1_train = scaler1.fit_transform(X1_train)
X1_test = scaler1.transform(X1_test)

#create log reg model
model1 = LogisticRegression(max_iter=1000, class_weight="balanced")

#train model
model1.fit(X1_train, y_train)

#make predictions
pred1 = model1.predict(X1_test)

#get probabilities
prob1 = model1.predict_proba(X1_test)[:, 1]

#evaluate model
print("Model 1")
print("Accuracy:", accuracy_score(y_test, pred1))
print("F1 Score:", f1_score(y_test, pred1))
print("ROC-AUC:", roc_auc_score(y_test, prob1))




Model 1
Accuracy: 0.5436208991494532
F1 Score: 0.6258964143426294
ROC-AUC: 0.6103112802182141


### Model 2: Review count + stars

In [ ]:
#choose feature
X2 = data[["log_reviews", "stars"]]

#split into train/test sets
X2_train, X2_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

#scale features
scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_test = scaler2.transform(X2_test)

#create log reg model
model2 = LogisticRegression(max_iter=1000, class_weight="balanced")

#train model
model2.fit(X2_train, y_train)

#make predictions
pred2 = model2.predict(X2_test)

#get probabilities
prob2 = model2.predict_proba(X2_test)[:, 1]

#evaluate model
print("Model 2")
print("Accuracy:", accuracy_score(y_test, pred2))
print("F1 Score:", f1_score(y_test, pred2))
print("ROC-AUC:", roc_auc_score(y_test, prob2))



Model 2
Accuracy: 0.5667071688942892
F1 Score: 0.6527750730282376
ROC-AUC: 0.6147454220790144


### Model 3: review count + stars + sentiment + complaints

In [ ]:
#choose feature
X3 = data[["log_reviews", "stars", "avg_sentiment", "avg_complaints"]]

#split into train/test sets
X3_train, X3_test, y_train, y_test = train_test_split(
    X3, y, test_size=0.2, random_state=42, stratify=y
)

#scale features
scaler3 = StandardScaler()
X3_train = scaler3.fit_transform(X3_train)
X3_test = scaler3.transform(X3_test)

#create log reg model
model3 = LogisticRegression(max_iter=1000, class_weight="balanced")

#train model
model3.fit(X3_train, y_train)

#make predictions
pred3 = model3.predict(X3_test)

#get probabilities
prob3 = model3.predict_proba(X3_test)[:, 1]

#evaluate model
print("Model 3")
print("Accuracy:", accuracy_score(y_test, pred3))
print("F1 Score:", f1_score(y_test, pred3))
print("ROC-AUC:", roc_auc_score(y_test, prob3))

Model 3
Accuracy: 0.6106925880923451
F1 Score: 0.7038817005545287
ROC-AUC: 0.654946604940035


Model performance improved as additional features were added. Review count alone provided the strongest baseline, while star ratings contributed a small improvement. Including sentiment resulted in a slight increase in performance, indicating that sentiment provides limited but meaningful additional predictive value. While complaint frequency is statistically significant, it does not substantially improve model performance, indicating that it is a weak predictor of operating status when used alongside other features.

Although sentiment showed a weak individual relationship with operating status, it contributed modest improvements when combined with other features.

### Model 4: everything + ubereats features

In [ ]:
import pandas as pd
import numpy as np

ubereats = pd.read_csv("/content/drive/MyDrive/163/ubereats.csv")

ubereats.head()

,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng
0,1,19,PJ Fresh (224 Daniel Payne Drive),NaN,NaN,"Burgers, American, Sandwiches",$,"224 Daniel Payne Drive, Birmingham, AL, 35207",35207,33.562365,-86.830703
1,2,9,J' ti`'z Smoothie-N-Coffee Bar,NaN,NaN,"Coffee and Tea, Breakfast and Brunch, Bubble Tea",NaN,"1521 Pinson Valley Parkway, Birmingham, AL, 35217",35217,33.583640,-86.773330
2,3,6,Philly Fresh Cheesesteaks (541-B Graymont Ave),NaN,NaN,"American, Cheesesteak, Sandwiches, Alcohol",$,"541-B Graymont Ave, Birmingham, AL, 35204",35204,33.509800,-86.854640
3,4,17,Papa Murphy's (1580 Montgomery Highway),NaN,NaN,Pizza,$,"1580 Montgomery Highway, Hoover, AL, 35226",35226,33.404439,-86.806614
4,5,162,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches",NaN,"314 17th St N, Birmingham, AL, 35203",35203,33.514730,-86.811700


In [ ]:
# Keep only the UberEats columns that we want to use in our analysis.
ubereats = ubereats[[
    "name",
    "score",
    "ratings",
    "price_range",
    "category",
    "zip_code",
    "lat",
    "lng"
]].copy()

In [ ]:
# Rename UberEats columns so they are clearly distinguished from Yelp columns.
# This prevents name conflicts after merging.
ubereats = ubereats.rename(columns={
    "name": "ubereats_name",
    "score": "ubereats_score",
    "ratings": "ubereats_ratings",
    "category": "ubereats_category",
    "lat": "ubereats_lat",
    "lng": "ubereats_lng"
})

In [ ]:
# Rename Yelp's postal_code column to zip_code so both datasets use the same name.
restaurants = restaurants.rename(columns={"postal_code": "zip_code"})

In [ ]:
# Select the Yelp columns needed for analysis and merging.
# This creates a clean restaurant-level dataset.
yelp = restaurants[[
    "business_id",
    "name",
    "zip_code",
    "stars",
    "review_count",
    "log_reviews",
    "avg_sentiment",
    "avg_complaints",
    "is_open"
]].copy()

In [ ]:
# Import regular expressions for text cleaning.
import re

# Function to standardize restaurant names before matching.
def clean_name(name):
  # Convert to lowercase.
    name = str(name).lower()
    # Remove punctuation and special characters.
    # Example: "Romano's Grill!" -> "romanos grill"
    name = re.sub(r"[^a-z0-9\s]", "", name)
    # Replace multiple spaces with a single space and trim.
    name = re.sub(r"\s+", " ", name).strip()
    return name

# Create standardized names in both datasets.
# This improves matching when punctuation or capitalization differs.
yelp["clean_name"] = yelp["name"].apply(clean_name)
ubereats["clean_name"] = ubereats["ubereats_name"].apply(clean_name)

# Standardize ZIP codes by converting them to strings
# and keeping only the first five digits.
yelp["zip_code"] = yelp["zip_code"].astype(str).str[:5]
ubereats["zip_code"] = ubereats["zip_code"].astype(str).str[:5]

merging on zipcode + name resulted in 0 matches -> merge only on name

In [ ]:

# Merge Yelp and UberEats using the cleaned restaurant name.
# We use a left join so every Yelp restaurant is kept,
# even if no UberEats match is found.merged = yelp.merge
 (
    ubereats,
    on="clean_name",
    how="left"
)

merged.head()

,business_id,name,zip_code_x,stars,review_count,log_reviews,avg_sentiment,avg_complaints,is_open,clean_name,ubereats_name,ubereats_score,ubereats_ratings,price_range,ubereats_category,zip_code_y,ubereats_lat,ubereats_lng
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,19107,4.0,80,4.394449,0.280990,0.252874,1,st honore pastries,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,33771,4.5,100,4.615121,0.294617,0.169811,0,zios italian market,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,19106,4.0,245,5.505332,0.343521,0.412000,1,tuna bar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ROeacJQwBeh05Rqg7F6TCg,BAP,19147,4.5,205,5.327876,0.298164,0.139423,1,bap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9OG5YkX1g2GReZM0AskizA,Romano's Macaroni Grill,89502,2.5,339,5.828946,0.134757,0.882022,1,romanos macaroni grill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Delivery presence variable

In [ ]:
# Create a binary feature indicating whether the restaurant
# was found in the UberEats dataset.
# 1 = present on UberEats
# 0 = not found
merged["delivery_presence"] = merged["ubereats_score"].notna().astype(int)

In [ ]:
# Replace missing UberEats ratings with 0.
# Missing values mean the restaurant was not matched.
merged["ubereats_score"] = merged["ubereats_score"].fillna(0)
merged["ubereats_ratings"] = merged["ubereats_ratings"].fillna(0)

# Convert price range symbols into numeric values.
# $    -> 1
# $$   -> 2
# $$$  -> 3
# $$$$ -> 4
price_map = {"$":1, "$$":2, "$$$":3, "$$$$":4}
merged["price_level"] = merged["price_range"].map(price_map).fillna(0)

In [ ]:
# Remove duplicate Yelp businesses.
# A restaurant may match multiple UberEats rows, so we keep
# only one row per Yelp business_id.
merged = merged.drop_duplicates(subset="business_id")

In [ ]:
X4 = merged[[
    "log_reviews",
    "stars",
    "avg_sentiment",
    "avg_complaints",
    "delivery_presence",
    "ubereats_score",
    "ubereats_ratings",
    "price_level"
]].dropna()

y4 = merged["is_open"]

In [ ]:
#split into train/test sets
X4_train, X4_test, y_train, y_test = train_test_split(
    X4, y4, test_size=0.2, random_state=42, stratify=y4
)

#scale features
scaler4 = StandardScaler()
X4_train = scaler4.fit_transform(X4_train)
X4_test = scaler4.transform(X4_test)

#create log reg model
model4 = LogisticRegression(max_iter=1000, class_weight="balanced")

#train model
model4.fit(X4_train, y_train)

#make predictions
pred4 = model4.predict(X4_test)

#get probabilities
prob4 = model4.predict_proba(X4_test)[:, 1]

#evaluate model
print("Model 4")
print("Accuracy:", accuracy_score(y_test, pred4))
print("F1 Score:", f1_score(y_test, pred4))
print("ROC-AUC:", roc_auc_score(y_test, prob4))

Model 4
Accuracy: 0.6051032806804374
F1 Score: 0.6975618834915317
ROC-AUC: 0.655755322842853


## Model 5: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X4_train_raw, X4_test_raw, y_train, y_test = train_test_split(
    X4, y4, test_size=0.2, random_state=42, stratify=y4
)


rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

rf.fit(X4_train_raw, y_train)

pred_rf = rf.predict(X4_test_raw)
prob_rf = rf.predict_proba(X4_test_raw)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, pred_rf))
print("F1 Score:", f1_score(y_test, pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, prob_rf))

Random Forest
Accuracy: 0.5978128797083839
F1 Score: 0.6842205685937798
ROC-AUC: 0.6581248218630601


Incorporating UberEats features resulted in a slight improvement in ROC-AUC, indicating a slight increase in the model’s ability to distinguish between open and closed restaurants.

In [ ]:
#!pip install -q kaleido
from google.colab import files

data = pd.DataFrame({
    "Model": ["Model 1", "Model 2", "Model 3", "Model 4"],
    "Accuracy": [0.54, 0.56, 0.61, 0.61],
    "F1": [0.62, 0.65, 0.70, 0.70],
    "ROC-AUC": [0.61, 0.61, 0.65, 0.66]
})

data_melted = data.melt(id_vars="Model", var_name="Metric", value_name="Score")

fig = px.bar(
    data_melted,
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    title="Model Comparison Across Metrics"
)

fig.show()

fig.write_html("model_comparison.html")
files.download("model_comparison.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import plotly.express as px
import pandas as pd

data = pd.DataFrame({
    "Model": ["Model 1", "Model 2", "Model 3", "Model 4"],
    "ROC-AUC": [0.61, 0.61, 0.65, 0.66]
})

fig = px.bar(
    data,
    x="Model",
    y="ROC-AUC",
    title="Model Comparison (ROC-AUC)",
    text="ROC-AUC"
)

fig.show()

The moderate performance across all models suggests that operating status is influenced by factors not captured in the dataset, such as location, cost, and competition.